***Setup and Loading***

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from typing import cast

# 1. Configurable project root via environment variable, defaulting to current working directory
PROJECT_ROOT = os.environ.get('AI_PIPELINE_ROOT', os.getcwd())
dataset_path = os.path.join(PROJECT_ROOT, 'datasets', 'blink_dataset.csv')

# 2. Explicit existence check with a clear error
if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        f"\n[CRITICAL ERROR] Dataset CSV is missing!\n"
        f"Looked in: {dataset_path}\n"
        f"Fix: Set the 'AI_PIPELINE_ROOT' environment variable to your project directory, "
        f"or ensure your Jupyter server is running inside the 'ai_pipeline' folder."
    )

print(f"Loading raw dataset from: {dataset_path}")

# 3. Load data with type hinting fixes using typing.cast instead of fragile asserts
raw_data = pd.read_csv(dataset_path)  # type: ignore
df = cast(pd.DataFrame, raw_data)

print("\n--- RAW DATA COUNTS ---")
print(df['Label'].value_counts().sort_index())

**The Automated Data Cleaner**

In [ ]:
threshold = 0.22

# --- THE CLEANING RULES ---
# Rule for Left Wink (1): Left eye MUST be closed, Right eye MUST be open
condition_left_wink = (df['Label'] == 1) & ((df['Left_EAR'] > threshold) | (df['Right_EAR'] < threshold))

# Rule for Right Wink (2): Right eye MUST be closed, Left eye MUST be open
condition_right_wink = (df['Label'] == 2) & ((df['Right_EAR'] > threshold) | (df['Left_EAR'] < threshold))

# Rule for Sustained Closure (3): BOTH eyes MUST be closed
condition_closure = (df['Label'] == 3) & ((df['Left_EAR'] > threshold) | (df['Right_EAR'] > threshold))

# Identify all dirty transition frames
dirty_frames = condition_left_wink | condition_right_wink | condition_closure

# Reassign the dirty frames back to Neutral (0)
df.loc[dirty_frames, 'Label'] = 0

print(f"Total transition frames safely moved to Neutral: {dirty_frames.sum()}")
print("\n--- CLEAN DATA COUNTS ---")
print(df['Label'].value_counts().sort_index())

*Training the Random Forest*

In [ ]:
# Define Features (X) and Labels (y)
# Timestamp is dropped because time of day doesn't affect a blink
X = df[['Left_EAR', 'Left_Min_15f', 'Left_Var_15f',
        'Right_EAR', 'Right_Min_15f', 'Right_Var_15f',
        'BoundingBox_Area']]
y = df['Label']

# Split the data: 80% for training, 20% for testing the accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training Random Forest Classifier...")
# max_depth=10 prevents overfitting and keeps the model execution blazing fast
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# Test the model
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"--- MODEL ACCURACY: {accuracy * 100:.2f}% ---")

**Exporting for C++ Integration**

In [ ]:
import os
import joblib

# Resolve relative to the same explicit project root
PROJECT_ROOT = os.environ.get('AI_PIPELINE_ROOT', os.getcwd())
model_output_path = os.path.join(PROJECT_ROOT, 'gesture_model.pkl')

# Save the compiled model so the C++ engine can use it
joblib.dump(clf, model_output_path)
print(f"Production Model successfully saved to:\n{model_output_path}")